### CSV And Excel Files - Structured Data 

In [1]:
import pandas as pd
import os 

In [3]:
os.makedirs("data/csv_files",exist_ok=True)

In [4]:
# Create sample data
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam'],
    
    'Category': ['Electronics', 'Accessories', 'Accessories', 
                 'Electronics', 'Electronics'],
    
    'Price': [999.99, 29.99, 79.99, 299.99, 89.99],
    
    'Stock': [50, 200, 150, 75, 100],
    
    'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}

# Save as CSV
df = pd.DataFrame(data)
df.to_csv("data/csv_files/products.csv", index=False)

print("CSV file created successfully!")
print(df)

CSV file created successfully!
    Product     Category   Price  Stock  \
0    Laptop  Electronics  999.99     50   
1     Mouse  Accessories   29.99    200   
2  Keyboard  Accessories   79.99    150   
3   Monitor  Electronics  299.99     75   
4    Webcam  Electronics   89.99    100   

                                         Description  
0  High-performance laptop with 16GB RAM and 512G...  
1       Wireless optical mouse with ergonomic design  
2          Mechanical keyboard with RGB backlighting  
3                27-inch 4K monitor with HDR support  
4               1080p webcam with noise cancellation  


In [6]:
# Save as Excel with multiple sheets

with pd.ExcelWriter('data/csv_files/inventory.xlsx') as writer:
    
    df.to_excel(writer, sheet_name='Products', index=False)

    # Add another sheet
    summary_data = {
        'Category': ['Electronics', 'Accessories'],
        'Total_Items': [3, 2],
        'Total_Value': [1389.97, 109.98]
    }

    pd.DataFrame(summary_data).to_excel(
        writer,
        sheet_name='Summary',
        index=False
    )

### CSV Processing

In [11]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [12]:
# Method 1: CSV loader - Each row becomes a document

print("CSV Loader - Row-Based Documents")

csv_loader = CSVLoader(file_path="data/csv_files/products.csv",
                    encoding="utf-8",csv_args={"delimiter": ",", "quotechar": '"'})

csv_docs =csv_loader.load()
print(f"Loader{len(csv_docs)} documents loaded.")
print("\nFIrst document:")
print(f"Content: {csv_docs[0].page_content}")
print(f"Metadata: {csv_docs[0].metadata}")

print("\n---\n")
## chunking 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

chunked_docs = text_splitter.split_documents(csv_docs)

print(f"\nTotal chunks created: {len(chunked_docs)}")

print("\nFirst Chunk:")
print(chunked_docs[0].page_content)

print("\nChunk Metadata:")
print(chunked_docs[0].metadata)

CSV Loader - Row-Based Documents
Loader5 documents loaded.

FIrst document:
Content: Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD
Metadata: {'source': 'data/csv_files/products.csv', 'row': 0}

---


Total chunks created: 10

First Chunk:
Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50

Chunk Metadata:
{'source': 'data/csv_files/products.csv', 'row': 0}


In [17]:
# MEthod 2: Custom CSV Processing for better control
from typing import List
from langchain_core.documents import Document

print("\nCustom CSV Processing ")
def process_csv_intelligently(filepath:str)-> List[Document]:
    """Process_csv with intelligent document creation"""
    df = pd.read_csv(filepath)
    document =[]
    
    #Strategy1 : One document per  row with structured content
    
    for idx, row in df.iterrows():
        #create structured content
        content = f""" Product Information:
        Name: {row['Product']}
        Category: {row['Category']}
        Price: ${row['Price']}
        Stock: {row['Stock']} units
        Description: {row['Description']}
        """
        
        # create document with rich metadata
        
        docs = Document(
            page_content = content,
            metadata ={
                "source": filepath,
                "row_index": idx,
                "product_name": row['Product'],
                "category": row['Category'],
                "price": row['Price'],
                "stock": row['Stock']
            }
        )
        document.append(docs)
        
    return document
        



Custom CSV Processing 


In [18]:
process_csv_intelligently("data/csv_files/products.csv")

[Document(metadata={'source': 'data/csv_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'stock': 50}, page_content=' Product Information:\n        Name: Laptop\n        Category: Electronics\n        Price: $999.99\n        Stock: 50 units\n        Description: High-performance laptop with 16GB RAM and 512GB SSD\n        '),
 Document(metadata={'source': 'data/csv_files/products.csv', 'row_index': 1, 'product_name': 'Mouse', 'category': 'Accessories', 'price': 29.99, 'stock': 200}, page_content=' Product Information:\n        Name: Mouse\n        Category: Accessories\n        Price: $29.99\n        Stock: 200 units\n        Description: Wireless optical mouse with ergonomic design\n        '),
 Document(metadata={'source': 'data/csv_files/products.csv', 'row_index': 2, 'product_name': 'Keyboard', 'category': 'Accessories', 'price': 79.99, 'stock': 150}, page_content=' Product Information:\n        Name: Keyboard\n        Categ

In [1]:
# 📊 CSV Processing Strategies

print("\n📊 CSV Processing Strategies:")
print("\n1. Row-based (CSVLoader):")

print("   ✅ Simple one-row-one-document")
print("   ✅ Good for record lookups")
print("   ❌ Loses table context")

print("\n2. Intelligent Processing:")

print("   ✅ Preserves relationships")
print("   ✅ Creates summaries")
print("   ✅ Rich metadata")
print("   ✅ Better for Q&A")


📊 CSV Processing Strategies:

1. Row-based (CSVLoader):
   ✅ Simple one-row-one-document
   ✅ Good for record lookups
   ❌ Loses table context

2. Intelligent Processing:
   ✅ Preserves relationships
   ✅ Creates summaries
   ✅ Rich metadata
   ✅ Better for Q&A


In [5]:
# Excel Processing
# Method 1: Using Pandas for full control
import pandas as pd
from typing import List
from langchain_core.documents import Document

print("📘 Pandas-based Excel Processing")

def process_excel_with_pandas(filepath: str) -> List[Document]:
    """Process Excel with sheet awareness"""

    documents = []

    # Read all sheets
    excel_file = pd.ExcelFile(filepath)

    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(filepath, sheet_name=sheet_name)

        # Create document for each sheet
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)

        doc = Document(
            page_content=sheet_content,
            metadata={
                'source': filepath,
                'sheet_name': sheet_name,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_type': 'excel_sheet'
            }
        )

        documents.append(doc)

    return documents

📘 Pandas-based Excel Processing


In [7]:
excel_docs = process_excel_with_pandas("data/csv_files/inventory.xlsx")
print(f"Processed {len(excel_docs)} sheets")

Processed 2 sheets


In [8]:
excel_docs

[Document(metadata={'source': 'data/csv_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: Products\nColumns: Product, Category, Price, Stock, Description\nRows: 5\n\n Product    Category  Price  Stock                                         Description\n  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD\n   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design\nKeyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting\n Monitor Electronics 299.99     75                 27-inch 4K monitor with HDR support\n  Webcam Electronics  89.99    100                1080p webcam with noise cancellation'),
 Document(metadata={'source': 'data/csv_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}, page_content='Sheet: Summary\nColumns: Category, Total_Items, Total_Valu

In [10]:
# Method 2 Unstructured Excel Loader
from langchain_community.document_loaders import UnstructuredExcelLoader
print("\n📘 Unstructured Excel Loader")

try:
    excel_loader = UnstructuredExcelLoader("data/csv_files/inventory.xlsx", mode = "elements")
    
    excel_docs = excel_loader.load()
    print(f"Loaded {len(excel_docs)} documents from Excel.")
except Exception as e:
    print(f"Error loading Excel: {e}")
    
    


📘 Unstructured Excel Loader
Loaded 2 documents from Excel.


In [11]:
excel_docs = process_excel_with_pandas("data/csv_files/inventory.xlsx")

In [12]:
excel_docs

[Document(metadata={'source': 'data/csv_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: Products\nColumns: Product, Category, Price, Stock, Description\nRows: 5\n\n Product    Category  Price  Stock                                         Description\n  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD\n   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design\nKeyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting\n Monitor Electronics 299.99     75                 27-inch 4K monitor with HDR support\n  Webcam Electronics  89.99    100                1080p webcam with noise cancellation'),
 Document(metadata={'source': 'data/csv_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}, page_content='Sheet: Summary\nColumns: Category, Total_Items, Total_Valu